# 向量动物园 · 端到端 demo

这个 notebook 把「两个词 → 基因组 → seekdb → 图鉴」整条链路跑一遍，并打印每一步真正执行的 SQL。

前置：seekdb 已在 `127.0.0.1:2881` 运行（`/seekdb_workspace/scripts/start_observer.sh`）。


In [ ]:
import os, sys, time, json
os.environ.setdefault("LLM_MODEL", "Qwen/Qwen2.5-7B-Instruct")
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "service")))

from zoo.db import ZooDB
from zoo.domain import ZooKeeper
from zoo.llm import LLM
from zoo import genome as G

db = ZooDB()
db.init_schema()
print(json.dumps(db.ping(), ensure_ascii=False, indent=2))
keeper = ZooKeeper(db=db, llm=LLM())
print(keeper.llm.warmup())


## 1. 造一只：`西瓜 × 章鱼`

`forge` 的数学部分（embedding 混合 + 突变 + KNN）不依赖 LLM，可以先单独看。

In [ ]:
a, b = "西瓜", "章鱼"
va, vb = G.embed([a, b])
g = G.mix(va, vb, ratio=0.5, mutation=0.08, seed=f"{a}|{b}")
print("维度", len(g), "前 8 维", [round(float(x), 4) for x in g[:8]])
print("基因轴", G.axes_of(g.tolist()))
print("指纹", G.gene_digest(g.tolist()))
kin = db.nearest(g.tolist(), k=5)
for k in kin:
    print(f"  距离 {k['dist']:.4f}  {k['code']} {k['name']}")


## 2. 让 LLM 写图鉴并落库

等价于 Dify 里的 ⑤ 撰写图鉴 → ⑦ 落库。

In [ ]:
t = time.time()
rec = keeper.synthesize("月亮", "榴莲", force=True)
print("耗时", round(time.time() - t, 1), "秒")
print(rec["code"], rec["name"], rec["rarity"])
print(rec["tagline"])
print(rec["lore"])


## 3. 混合检索：向量 + 全文 + JSON 过滤，一条 SQL

In [ ]:
res = db.hybrid_search("会飞的", G.embed(["会飞的"])[0], k=6)
print(res["sql"])
print()
for x in res["results"]:
    print(f"{x['score']:.4f}  vec={1-x['dist']:.3f} text={x['text_score']:.3f}  {x['code']} {x['name']}")


## 4. 生态：全园统计 + 基因雷达

In [ ]:
st = db.stats()
print(json.dumps(st, ensure_ascii=False, indent=2))

import matplotlib.pyplot as plt
import numpy as np
r = db.list_creatures(limit=1)[0]
axes = r["axes"]
labels = list(axes.keys())
vals = [axes[k] / 100 for k in labels]
ang = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
vals += vals[:1]; ang += ang[:1]
fig = plt.figure(figsize=(3.6, 3.6))
ax = fig.add_subplot(111, polar=True)
ax.plot(ang, vals, color="#e9b45b")
ax.fill(ang, vals, color="#e9b45b", alpha=0.25)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(labels, fontsize=9)
ax.set_yticklabels([]); ax.set_title(r["name"], fontsize=12, pad=14)
plt.show()


## 5. 同一套流程搬进 Dify

见 `dify/vector-zoo.dsl.yml`：9 个节点，把上面这几步拆成 LLM / Code / HTTP 节点。
其中 ④ 与 ⑦ 对应本 notebook 的 `forge` 与 `commit` 两个接口。